# 07 — Comparación topológica entre regiones (Mejora 2)

## ¿Qué hacemos aquí?

El notebook 03 comparó CDMX y EDOMEX con distancias bottleneck/Wasserstein — útil para dos regiones. Para comparar **N estados simultáneamente** necesitamos convertir cada diagrama en un **vector de longitud fija** que podamos comparar con similitud coseno, PCA o clustering jerárquico.

Técnica: **Persistence Images** (persim). Cada hueco H₁ aporta una gaussiana centrada en su posición (birth, persistencia) sobre un grid compartido. La imagen resultante codifica *dónde se concentran los huecos* en el espacio topológico.

## Filtro de huecos significativos

El diagrama H₁ completo tiene ~13k–19k clases, la mayoría de persistencia < 1 m (artefactos numéricos). Solo trabajamos con los **71 (CDMX) y 209 (EDOMEX)** huecos con persistencia ≥ 200 m que identificamos en el notebook 03.

In [ ]:
import sys
from pathlib import Path
_ROOT = Path("..").resolve()
sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyproj import Transformer
from lib import data, tda, vectorizar, config

## Cálculo de huecos y vectorización

In [ ]:
REGIONES     = ["CDMX", "EDOMEX"]
MIN_PERS_IMG = 200.0

print("Calculando diagramas H₁ (solo clases con persistencia >= 200 m)...")
diags_h1     = {}
diags_h1_sig = {}
for region in REGIONES:
    df = pd.read_parquet(config.INTERMEDIOS_DIR / f"salud_{region}.parquet")
    P  = data.puntos(df)
    st = tda.alpha_complex(P)
    dr = tda.a_radio(tda.persistencia(st))
    d  = dr[1]
    diags_h1[region] = d
    mask = np.isfinite(d[:, 1]) & ((d[:, 1] - d[:, 0]) >= MIN_PERS_IMG)
    diags_h1_sig[region] = d[mask]
    print(f"  {region}: {int(mask.sum())} huecos significativos (de {len(d)} totales)")

pimgr = vectorizar.ajustar_imager(list(diags_h1_sig.values()), pixel_size=500.0, sigma=800.0)
vectores = {r: vectorizar.vectorizar(diags_h1_sig[r], pimgr) for r in REGIONES}
mat_sim = vectorizar.matriz_similitud(vectores, REGIONES)
print(f"\nSimilitud coseno CDMX ↔ EDOMEX: {mat_sim[0,1]:.3f}")

## Panel comparativo

El panel tiene 4 gráficas:

1. **Scatter CDMX** — cada punto es un hueco, posición = (birth, persistencia), color = tamaño.
2. **Scatter EDOMEX** — ídem. Comparar visualmente la dispersión de puntos.
3. **Histograma** (escala log) — distribución de tamaños de huecos. La cola a la derecha de EDOMEX evidencia huecos enormes que CDMX no tiene.
4. **Boxplot** — la mediana, rango intercuartil y outliers muestran la diferencia estructural.

**Similitud coseno = 0.635 / 1.0** — con los huecos reales filtrados (no ruido), la distancia entre ambas regiones es de 0.365, indicando estructuras de cobertura notablemente distintas.

In [ ]:
ruta, _ = vectorizar.panel_vectorizacion(pimgr, diags_h1_sig, vectores, REGIONES)
print(f"Figura: {ruta}")